# Distil an LLM planning agent into a 0.5B model and serve it efficiently

**Runtime:** GPU (L4 or A100 recommended; a T4 works for the 0.5B student but use a 3B teacher). About 1-2 hours end to end.

Steps
1. Serve an open **teacher** (Qwen2.5-7B-Instruct) with vLLM and record its planning calls on many studies.
2. Curate the recorded calls into an SFT set (only good, guard-clean plans from low-regret studies).
3. LoRA fine-tune **Qwen2.5-0.5B-Instruct** on it, merge the adapter.
4. Serve the distilled model with vLLM (fp16, then 4-bit bitsandbytes), measure latency / tokens/s / JSON validity.
5. Benchmark teacher vs base-0.5B vs distilled-0.5B vs BO on the same tasks and seeds (regret at fixed budget).

Every number this notebook prints is a measurement on your hardware - copy them into `bench/RESULTS_GPU.md`.

In [ ]:
!nvidia-smi
!git clone https://github.com/SyedaRubbani/sciagent-bench.git
%cd sciagent-bench
!pip -q install -e ".[dev]" vllm "transformers>=4.44" "peft>=0.12" accelerate bitsandbytes

## 1. Teacher traces

In [ ]:
import subprocess
import time

teacher = subprocess.Popen("vllm serve Qwen/Qwen2.5-7B-Instruct --port 8001 --max-model-len 16384 --gpu-memory-utilization 0.85".split(),
                           stdout=open("vllm_teacher.log","w"), stderr=subprocess.STDOUT)
import httpx

for _ in range(120):
    try:
        httpx.get("http://localhost:8001/v1/models", timeout=2); break
    except Exception: time.sleep(5)
print(open("vllm_teacher.log").read()[-1500:])

In [ ]:
# ~3 tasks x 12 seeds x 2 budgets of recorded studies
!sciagent bench --planners recipes/distill_planner_lora/teacher.yaml --seeds 12 --budget 300 --keep-runs --out teacher_runs/b300
!sciagent bench --planners recipes/distill_planner_lora/teacher.yaml --seeds 12 --budget 150 --keep-runs --out teacher_runs/b150

## 2. Curate the SFT set

In [ ]:
!mkdir -p teacher_runs/all && cp -r teacher_runs/b300/runs/* teacher_runs/b150/runs/* teacher_runs/all/
!python recipes/distill_planner_lora/make_sft_data.py teacher_runs/all --out sft.jsonl --keep-frac 0.7
!wc -l sft.jsonl sft.val.jsonl

## 3. LoRA fine-tune the 0.5B student (stop the teacher first to free GPU memory)

In [ ]:
teacher.terminate(); teacher.wait()
!python recipes/distill_planner_lora/train_lora.py --data sft.jsonl --base Qwen/Qwen2.5-0.5B-Instruct --out planner-lora --epochs 3

## 4. Serve and measure inference efficiency

In [ ]:
def serve(cmd, port, log):
    p = subprocess.Popen(cmd.split(), stdout=open(log,"w"), stderr=subprocess.STDOUT)
    for _ in range(120):
        try:
            httpx.get(f"http://localhost:{port}/v1/models", timeout=2); return p
        except Exception: time.sleep(5)
    raise RuntimeError(open(log).read()[-2000:])

student = serve("vllm serve planner-lora/merged --served-model-name sciagent-planner-0.5b --port 8002 --max-model-len 16384 --gpu-memory-utilization 0.3", 8002, "vllm_student.log")
!python recipes/distill_planner_lora/bench_inference.py --base-url http://localhost:8002/v1 --model sciagent-planner-0.5b --data sft.val.jsonl

In [ ]:
# 4-bit weights via bitsandbytes in vLLM (older vLLM versions also need: --load-format bitsandbytes)
student.terminate(); student.wait()
student4 = serve("vllm serve planner-lora/merged --served-model-name sciagent-planner-0.5b --port 8002 --quantization bitsandbytes --max-model-len 16384 --gpu-memory-utilization 0.3", 8002, "vllm_student4.log")
!python recipes/distill_planner_lora/bench_inference.py --base-url http://localhost:8002/v1 --model sciagent-planner-0.5b --data sft.val.jsonl
student4.terminate(); student4.wait()

## 5. Planning quality: teacher vs base vs distilled vs BO

In [ ]:
teacher = serve("vllm serve Qwen/Qwen2.5-7B-Instruct --port 8001 --max-model-len 16384 --gpu-memory-utilization 0.55", 8001, "vllm_teacher2.log")
student = serve("vllm serve planner-lora/merged --served-model-name sciagent-planner-0.5b --port 8002 --max-model-len 16384 --gpu-memory-utilization 0.15", 8002, "vllm_student2.log")
base = serve("vllm serve Qwen/Qwen2.5-0.5B-Instruct --port 8003 --max-model-len 16384 --gpu-memory-utilization 0.15", 8003, "vllm_base.log")
# held-out seeds (100+) so the student is not scored on studies it was trained on
!sciagent bench --planners recipes/distill_planner_lora/student.yaml --seeds 5 --seed-start 100 --budget 300 --out gpu_results
print(open("gpu_results/results.md").read())